In [ ]:
# Estudiante: Masiel Aguilar Ameller
# Codigo: 87770

import pandas as pd
import numpy as np
from datetime import datetime
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen
import warnings
warnings.filterwarnings('ignore')

class BBBPReportGenerator:
    def __init__(self, csv_path):
        """
        Initialize the BBBP Report Generator
        
        Args:
            csv_path (str): Path to the BBBP.csv file
        """
        self.csv_path = csv_path
        self.data = None
        self.load_data()
        
    def load_data(self):
        """Load the BBBP dataset from CSV file"""
        try:
            self.data = pd.read_csv(self.csv_path)
            print(f"Dataset loaded successfully: {len(self.data)} compounds")
            print(f"Columns: {list(self.data.columns)}")
        except Exception as e:
            print(f"Error loading dataset: {e}")
            
    def calculate_molecular_properties(self, smiles):
        """
        Calculate molecular properties from SMILES string
        
        Args:
            smiles (str): SMILES representation of the molecule
            
        Returns:
            dict: Dictionary containing molecular properties
        """
        try:
            # Suppress RDKit warnings temporarily
            from rdkit import RDLogger
            RDLogger.DisableLog('rdApp.*')
            
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return self.get_default_properties()
            
            properties = {
                'molecular_weight': round(Descriptors.MolWt(mol), 2),
                'logp': round(Crippen.MolLogP(mol), 2),
                'hbd': Descriptors.NumHDonors(mol),  # Hydrogen bond donors
                'hba': Descriptors.NumHAcceptors(mol),  # Hydrogen bond acceptors
                'tpsa': round(Descriptors.TPSA(mol), 2),  # Topological polar surface area
                'rotatable_bonds': Descriptors.NumRotatableBonds(mol),
                'aromatic_rings': rdMolDescriptors.CalcNumAromaticRings(mol),
                'heavy_atoms': Descriptors.HeavyAtomCount(mol),
                'complexity': round(Descriptors.BertzCT(mol), 2)
            }
            
            # Lipinski's Rule of Five compliance
            properties['lipinski_violations'] = sum([
                properties['molecular_weight'] > 500,
                properties['logp'] > 5,
                properties['hbd'] > 5,
                properties['hba'] > 10
            ])
            
            return properties
            
        except Exception as e:
            # Silent handling of RDKit errors
            return self.get_default_properties()
    
    def get_default_properties(self):
        """Return default properties when calculation fails"""
        return {
            'molecular_weight': 'N/A',
            'logp': 'N/A',
            'hbd': 'N/A',
            'hba': 'N/A',
            'tpsa': 'N/A',
            'rotatable_bonds': 'N/A',
            'aromatic_rings': 'N/A',
            'heavy_atoms': 'N/A',
            'complexity': 'N/A',
            'lipinski_violations': 'N/A'
        }
    
    def assess_bbb_penetration_factors(self, properties, p_np):
        """
        Assess factors affecting blood-brain barrier penetration
        
        Args:
            properties (dict): Molecular properties
            p_np (int): Penetration label (1 for penetrant, 0 for non-penetrant)
            
        Returns:
            dict: Assessment of BBB penetration factors
        """
        assessment = {
            'penetration_status': 'Penetrant' if p_np == 1 else 'Non-penetrant',
            'favorable_factors': [],
            'unfavorable_factors': [],
            'overall_assessment': '',
            'confidence_level': 'Medium'
        }
        
        if properties['molecular_weight'] != 'N/A':
            # Molecular weight assessment
            if properties['molecular_weight'] < 400:
                assessment['favorable_factors'].append('Low molecular weight (< 400 Da) favors BBB penetration')
            elif properties['molecular_weight'] > 500:
                assessment['unfavorable_factors'].append('High molecular weight (> 500 Da) hinders BBB penetration')
            
            # LogP assessment
            if properties['logp'] != 'N/A':
                if 1 <= properties['logp'] <= 3:
                    assessment['favorable_factors'].append('Optimal lipophilicity (LogP 1-3) for BBB penetration')
                elif properties['logp'] < 1:
                    assessment['unfavorable_factors'].append('Low lipophilicity may limit BBB penetration')
                elif properties['logp'] > 5:
                    assessment['unfavorable_factors'].append('High lipophilicity may cause non-specific binding')
            
            # TPSA assessment
            if properties['tpsa'] != 'N/A':
                if properties['tpsa'] < 60:
                    assessment['favorable_factors'].append('Low polar surface area (< 60 Ų) favors BBB penetration')
                elif properties['tpsa'] > 90:
                    assessment['unfavorable_factors'].append('High polar surface area (> 90 Ų) hinders BBB penetration')
            
            # Hydrogen bonding assessment
            if properties['hbd'] != 'N/A' and properties['hba'] != 'N/A':
                total_hb = properties['hbd'] + properties['hba']
                if total_hb <= 7:
                    assessment['favorable_factors'].append('Low hydrogen bonding potential favors BBB penetration')
                else:
                    assessment['unfavorable_factors'].append('High hydrogen bonding potential hinders BBB penetration')
        
        # Overall assessment
        favorable_count = len(assessment['favorable_factors'])
        unfavorable_count = len(assessment['unfavorable_factors'])
        
        if favorable_count > unfavorable_count:
            assessment['overall_assessment'] = 'Molecular properties generally favor BBB penetration'
            assessment['confidence_level'] = 'High' if favorable_count >= 3 else 'Medium'
        elif unfavorable_count > favorable_count:
            assessment['overall_assessment'] = 'Molecular properties generally hinder BBB penetration'
            assessment['confidence_level'] = 'High' if unfavorable_count >= 3 else 'Medium'
        else:
            assessment['overall_assessment'] = 'Mixed molecular properties - uncertain BBB penetration'
            assessment['confidence_level'] = 'Low'
        
        return assessment
    
    def generate_compound_report(self, index, compound_data):
        """
        Generate a detailed report for a single compound
        
        Args:
            index (int): Index of the compound
            compound_data (pandas.Series): Data for the compound
            
        Returns:
            str: Formatted report
        """
        name = compound_data.get('name', f'Compound_{index}')
        smiles = compound_data.get('smiles', '')
        p_np = compound_data.get('p_np', 0)
        
        # Calculate molecular properties
        properties = self.calculate_molecular_properties(smiles)
        
        # Assess BBB penetration factors
        assessment = self.assess_bbb_penetration_factors(properties, p_np)
        
        # Generate report
        report = f"""
{'='*80}
BLOOD-BRAIN BARRIER PENETRATION ANALYSIS REPORT
{'='*80}

Report ID: BBBP_{index:04d}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

COMPOUND IDENTIFICATION
{'─'*40}
Name: {name}
SMILES: {smiles}
Dataset Index: {index}

BLOOD-BRAIN BARRIER PENETRATION STATUS
{'─'*40}
Classification: {assessment['penetration_status']}
Confidence Level: {assessment['confidence_level']}

MOLECULAR PROPERTIES ANALYSIS
{'─'*40}
• Molecular Weight: {properties['molecular_weight']} Da
• Lipophilicity (LogP): {properties['logp']}
• Hydrogen Bond Donors: {properties['hbd']}
• Hydrogen Bond Acceptors: {properties['hba']}
• Topological Polar Surface Area: {properties['tpsa']} Ų
• Rotatable Bonds: {properties['rotatable_bonds']}
• Aromatic Rings: {properties['aromatic_rings']}
• Heavy Atoms: {properties['heavy_atoms']}
• Molecular Complexity: {properties['complexity']}
• Lipinski Rule Violations: {properties['lipinski_violations']}

BBB PENETRATION ASSESSMENT
{'─'*40}
Overall Assessment: {assessment['overall_assessment']}

Favorable Factors:"""
        
        if assessment['favorable_factors']:
            for factor in assessment['favorable_factors']:
                report += f"\n  ✓ {factor}"
        else:
            report += "\n  • No significant favorable factors identified"
        
        report += "\n\nUnfavorable Factors:"
        
        if assessment['unfavorable_factors']:
            for factor in assessment['unfavorable_factors']:
                report += f"\n  ✗ {factor}"
        else:
            report += "\n  • No significant unfavorable factors identified"
        
        # Add drug-likeness assessment
        report += f"""

DRUG-LIKENESS ASSESSMENT
{'─'*40}
Lipinski Rule of Five Compliance: {'PASS' if properties['lipinski_violations'] == 0 else f'FAIL ({properties["lipinski_violations"]} violations)'}

CLINICAL RELEVANCE
{'─'*40}
"""
        
        if p_np == 1:
            report += """This compound demonstrates blood-brain barrier penetration capability, making it:
• Potentially suitable for CNS drug development
• Likely to reach brain tissue after systemic administration
• Important to monitor for CNS side effects if used therapeutically
• Valuable for neurological and psychiatric drug research"""
        else:
            report += """This compound does not penetrate the blood-brain barrier effectively:
• Unlikely to cause CNS side effects
• Not suitable for brain-targeted therapeutic applications
• May be preferred for peripheral drug targets
• Could serve as a starting point for BBB penetration optimization"""
        
        report += f"""

RESEARCH RECOMMENDATIONS
{'─'*40}
"""
        
        if p_np == 1:
            report += """• Investigate CNS pharmacokinetics and brain distribution
• Evaluate potential neurological effects
• Consider for CNS disease therapeutic development
• Monitor for blood-brain barrier transporter interactions"""
        else:
            report += """• Consider structural modifications to enhance BBB penetration if CNS activity is desired
• Investigate peripheral pharmacokinetics
• Evaluate for non-CNS therapeutic applications
• Study as negative control in BBB penetration studies"""
        
        report += f"""

QUALITY CONTROL
{'─'*40}
Data Source: BBBP Dataset (Martins et al. 2012)
Analysis Method: RDKit molecular descriptors + Rule-based assessment
Report Status: {'COMPLETE' if properties['molecular_weight'] != 'N/A' else 'LIMITED (SMILES parsing failed)'}

{'='*80}
END OF REPORT
{'='*80}
"""
        
        return report
    
    def generate_reports(self, num_reports=200, output_dir='bbbp_reports'):
        """
        Generate reports for multiple compounds
        
        Args:
            num_reports (int): Number of reports to generate
            output_dir (str): Directory to save reports
        """
        if self.data is None:
            print("No data loaded. Please check the CSV file path.")
            return
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        
        # Limit to available data
        num_reports = min(num_reports, len(self.data))
        
        print(f"Generating {num_reports} reports...")
        
        # Generate summary statistics
        penetrant_count = sum(self.data['p_np'][:num_reports])
        non_penetrant_count = num_reports - penetrant_count
        
        # Generate individual reports
        for i in range(num_reports):
            try:
                report = self.generate_compound_report(i, self.data.iloc[i])
                
                # Save report to file - clean filename
                compound_name = self.data.iloc[i].get('name', f'Compound_{i}')
                # Remove problematic characters from filename
                clean_name = compound_name.replace('/', '_').replace('\\', '_').replace(':', '_').replace('*', '_').replace('?', '_').replace('"', '').replace('<', '_').replace('>', '_').replace('|', '_').replace(' ', '_')
                filename = f"BBBP_Report_{i:04d}_{clean_name}.txt"
                filepath = os.path.join(output_dir, filename)
                
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(report)
                
                if (i + 1) % 50 == 0:
                    print(f"Generated {i + 1} reports...")
                    
            except Exception as e:
                print(f"Error generating report for compound {i}: {e}")
        
        # Generate summary report
        summary_report = f"""
BBBP DATASET ANALYSIS SUMMARY
{'='*50}

Total Reports Generated: {num_reports}
BBB Penetrant Compounds: {penetrant_count} ({penetrant_count/num_reports*100:.1f}%)
Non-penetrant Compounds: {non_penetrant_count} ({non_penetrant_count/num_reports*100:.1f}%)

Dataset Information:
• Source: Blood-brain barrier penetration (BBBP) dataset
• Reference: Martins et al. (2012) J. Chem. Inf. Model.
• Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Reports saved in: {output_dir}/
"""
        
        with open(os.path.join(output_dir, 'SUMMARY_REPORT.txt'), 'w') as f:
            f.write(summary_report)
        
        print(f"\nCompleted! Generated {num_reports} individual reports + 1 summary report")
        print(f"All reports saved in '{output_dir}' directory")
        print(f"BBB Penetrant: {penetrant_count}, Non-penetrant: {non_penetrant_count}")

# Usage example
if __name__ == "__main__":
    # Initialize the report generator
    # Try different possible paths for the BBBP.csv file
    possible_paths = [
        "BBBP.csv",  # Same directory as script
        "bbbp/BBBP.csv",  # In bbbp subdirectory
        "../BBBP.csv",  # Parent directory
        "data/BBBP.csv",  # In data subdirectory
        os.path.join(os.getcwd(), "BBBP.csv"),  # Current working directory
    ]
    
    csv_path = None
    for path in possible_paths:
        if os.path.exists(path):
            csv_path = path
            print(f"Found BBBP.csv at: {csv_path}")
            break
    
    if csv_path is None:
        print("BBBP.csv file not found in common locations.")
        print("Please specify the correct path to your BBBP.csv file:")
        print("Current working directory:", os.getcwd())
        print("Files in current directory:", [f for f in os.listdir('.') if f.endswith('.csv')])
        csv_path = input("Enter the full path to BBBP.csv: ").strip()
    
    try:
        # Create report generator instance
        generator = BBBPReportGenerator(csv_path)
        
        # Generate 200 reports (or specify different number)
        generator.generate_reports(num_reports=200, output_dir='bbbp_reports')
        
        print("\n" + "="*60)
        print("BBBP REPORT GENERATION COMPLETED SUCCESSFULLY!")
        print("="*60)
        
    except Exception as e:
        print(f"Error running report generator: {e}")
        print("\nPlease ensure:")
        print("1. The BBBP.csv file exists in the specified path")
        print("2. Required libraries are installed: pip install pandas rdkit-pypi")
        print("3. The file path is correct for your system")

Found BBBP.csv at: BBBP.csv
Dataset loaded successfully: 2050 compounds
Columns: ['num', 'name', 'p_np', 'smiles']
Generating 200 reports...
Generated 50 reports...
Generated 100 reports...
Generated 150 reports...
Generated 200 reports...

Completed! Generated 200 individual reports + 1 summary report
All reports saved in 'bbbp_reports' directory
BBB Penetrant: 140, Non-penetrant: 60

BBBP REPORT GENERATION COMPLETED SUCCESSFULLY!
